In [15]:
from torchvision import datasets
import torchvision  

In [16]:
train_data = datasets.MNIST(
    root='data',
    train= True,
    transform= torchvision.transforms.ToTensor(),
    download= True
)


test_data = datasets.MNIST(
    root='data',
    train= False,
    transform= torchvision.transforms.ToTensor(),
    download= True
)


In [17]:
train_data.data

tensor([[[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        ...,

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]],

        [[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0,

In [18]:
train_data.data.shape

torch.Size([60000, 28, 28])

In [19]:
from torch.utils.data import DataLoader

loader = {
    'train' : DataLoader(train_data,
                         batch_size=100,
                         shuffle= True,
                         num_workers= 1),

    'test' : DataLoader(test_data,
                        batch_size=100,
                        shuffle= False,
                        num_workers=1)
}

In [20]:
loader

{'train': <torch.utils.data.dataloader.DataLoader at 0x284944e2840>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x284915e3f50>}

Defining NN Model

In [21]:
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim


In [22]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)

        self.drop_out = nn.Dropout2d()

        # Fixed here ↓
        self.lc1 = nn.Linear(320, 50)   # 320 comes from x.view(-1, 320)
        self.lc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.drop_out(self.conv2(x)), 2))

        x = x.view(-1, 320)          # This must match lc1 input size
        x = F.relu(self.lc1(x))
        x = F.dropout(x, training=self.training)
        x = self.lc2(x)

        return F.log_softmax(x, dim=1)   # Better than softmax + better numerical stability

In [23]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'CPU')

model = CNN().to(device)

optimizer = optim.Adam(model.parameters(),lr = 0.001)

loss_fn = nn.CrossEntropyLoss()


In [24]:
def train(epochs):
    model.train()

    for batch_idx, (data, target) in enumerate(loader['train']):
        data, target = data.to(device),target.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = loss_fn(output,target)

        loss.backward()

        optimizer.step()

        if batch_idx % 20 == 0:
            print(f"train epoch : {epochs} [{batch_idx * len(data)}/{len(loader['train'].dataset)}]")

In [28]:
def test():
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in loader['test']:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            
            test_loss += loss_fn(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(loader['test'].dataset)
    
    accuracy = 100. * correct / len(loader['test'].dataset)
    
    print(f'Test set: Average loss: {test_loss:.4f}, Accuracy: {accuracy:.2f}%')

In [29]:

for epoch in range(1,11):
    train(epoch)
    test()

train epoch : 1 [0/60000]
train epoch : 1 [2000/60000]
train epoch : 1 [4000/60000]
train epoch : 1 [6000/60000]
train epoch : 1 [8000/60000]
train epoch : 1 [10000/60000]
train epoch : 1 [12000/60000]
train epoch : 1 [14000/60000]
train epoch : 1 [16000/60000]
train epoch : 1 [18000/60000]
train epoch : 1 [20000/60000]
train epoch : 1 [22000/60000]
train epoch : 1 [24000/60000]
train epoch : 1 [26000/60000]
train epoch : 1 [28000/60000]
train epoch : 1 [30000/60000]
train epoch : 1 [32000/60000]
train epoch : 1 [34000/60000]
train epoch : 1 [36000/60000]
train epoch : 1 [38000/60000]
train epoch : 1 [40000/60000]
train epoch : 1 [42000/60000]
train epoch : 1 [44000/60000]
train epoch : 1 [46000/60000]
train epoch : 1 [48000/60000]
train epoch : 1 [50000/60000]
train epoch : 1 [52000/60000]
train epoch : 1 [54000/60000]
train epoch : 1 [56000/60000]
train epoch : 1 [58000/60000]
Test set: Average loss: 0.0008, Accuracy: 97.44%
train epoch : 2 [0/60000]
train epoch : 2 [2000/60000]
trai

In [30]:
torch.save(model.state_dict(), 'mnist_cnn.pth')